# Demo Pipeline — Legal Contract Analyzer

This notebook demonstrates the full end-to-end pipeline on a real contract.

## Contract chosen
We use a sample Romanian services contract (`data/contract_exemplu.pdf`).  
It was chosen because it contains several common risk patterns:
- Uncapped penalty clauses (RIDICAT)
- Missing GDPR legal basis for data processing (RIDICAT)
- Ambiguous force majeure definition (MEDIU)

## Expected behaviour
- DocumentParserAgent should extract 15+ clauses
- RAGRetrievalAgent should return 3–5 chunks per clause
- RiskAssessmentAgent should flag the penalty and GDPR clauses as RIDICAT
- RecommendationAgent should produce reformulations for both

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')  # Run from project root

from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(level=logging.WARNING)  # Keep output clean in notebook

## 1. Run the LangGraph pipeline

In [ ]:
from src.graph.workflow import run_pipeline, export_graph_png

CONTRACT_PATH = 'data/contract_exemplu.pdf'

# Run full pipeline
final_state = run_pipeline(CONTRACT_PATH)

print('Pipeline complete!')
print(f"Report: {final_state['report_path']}")
print(f"Iterations: {final_state['iteration']}")
print(f"High risk alert: {final_state['high_risk_alert']}")

## 2. Final state summary

In [ ]:
from src.dtos import RiskLevel

risk_map = final_state['risk_map']
counts = {level.value: 0 for level in RiskLevel}
for assessment in risk_map.values():
    counts[assessment.risk_level.value] += 1

print('Risk distribution:')
for level, count in counts.items():
    bar = '█' * count
    print(f'  {level:12s}: {bar} ({count})')

## 3. RAGAS Scores (commented)

In [ ]:
import json

with open('logs/rag_evaluation.json', 'r') as f:
    ragas = json.load(f)

print(f"Average faithfulness   : {ragas['avg_faithfulness']:.3f}")
print(f"Average answer relevancy: {ragas['avg_answer_relevancy']:.3f}")
print(f"Average context recall : {ragas['avg_context_recall']:.3f}")
print(f"Pass rate (>=0.6)      : {ragas['pass_rate']:.1%}")
print()
print("""
Interpretation:
- faithfulness >= 0.6 means the LLM is using retrieved chunks rather than hallucinating.
  In legal context this is critical — a faithfulness of 0.5 means ~half the statements
  have no grounding in the corpus, which is unacceptable.
- answer_relevancy >= 0.6 means the retrieved context actually answers the legal question.
  Low scores here usually mean the corpus is too narrow or chunking is too coarse.
- context_recall >= 0.6 means the chunks contain substantive legal text.
  Chunks from headers/footers inflate count without adding value.
""")

## 4. Visualizations

In [ ]:
from IPython.display import Image, display

print('Retrieval Heatmap (5 test clauses × top-3 chunks):')
display(Image('logs/retrieval_heatmap.png'))

In [ ]:
# Risk distribution chart
import matplotlib.pyplot as plt
import os

labels = list(counts.keys())
values = list(counts.values())
colors = ['#ff6b6b', '#ffd93d', '#fff9c4', '#6bcb77', '#e0e0e0']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=colors[:len(labels)], edgecolor='gray')
ax.set_title('Risk Distribution across all Contract Clauses')
ax.set_ylabel('Number of clauses')
for bar, val in zip(bars, values):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', va='bottom')
os.makedirs('logs', exist_ok=True)
plt.savefig('logs/risk_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
display(Image('logs/risk_distribution.png'))

In [ ]:
# LangGraph diagram
from src.graph.workflow import export_graph_png
export_graph_png('logs/workflow_graph.png')
display(Image('logs/workflow_graph.png'))

## 5. Final Report

In [ ]:
from IPython.display import Markdown, display

with open(final_state['report_path'], 'r', encoding='utf-8') as f:
    report_text = f.read()

display(Markdown(report_text))

## 6. Conclusion

In [ ]:
import json

# Find the most recent run log
import glob
logs = sorted(glob.glob('logs/run_*.json'), reverse=True)
if logs:
    with open(logs[0]) as f:
        run_log = json.load(f)
    
    print('Per-node timing:')
    total_time = 0
    for entry in run_log:
        dur = entry.get('duration_s', 0)
        total_time += dur
        print(f"  {entry['node']:30s}: {dur:.2f}s")
    print(f"  {'TOTAL':30s}: {total_time:.2f}s")

print()
print('Estimated cost (rough):')
print('  parse_document     : ~$0.002  (gpt-4o-mini, metadata extraction)')
print('  assess_risk        : ~$0.010  (gpt-4o-mini × num clauses)')
print('  recommendations    : ~$0.020  (gpt-4o-mini × risky clauses, 3-4x for RIDICAT)')
print('  TOTAL per contract : ~$0.03–0.05 typical')

print()
print('Known limitation identified:')
print('  Scanned PDFs (image-only) return empty text from pdfplumber.')
print('  Without OCR (pytesseract), these contracts cannot be analyzed.')
print('  Improvement: add OCR fallback using pytesseract + pdf2image.')